# Recaller List Pipeline

| | |
|---|---|
| **Owner** | Lisa |
| **Team** | Nordic Customer |
| **Last updated** | 2026-11-23 |
| **Schedule** | Manual (not scheduled yet) |
| **Output table** | cosdpprd.glall_user.recallerlist |
| **Scope** | First calls on and after 2026-05-01 (`start_datekey` = 20260501) |

## Purpose
One table of recallers: customers whose **first recall is within 0 to 30 days** of the first call (`RecallInXDays` between 0 and `recall_window_days`).
One row = one recaller = **BrandKey + hashed phone**. A customer with two phone numbers can appear twice.

## How it works
1. Parameters: widgets are bound to session variables.
2. Staging temp views (no writes): `rl_calls` -> `rl_recallers` -> `rl_contracts` -> `rl_status` -> `rl_primary_contract` / `rl_vas` -> `rl_payment` -> `rl_digital` -> `rl_final`.
3. `CREATE TABLE IF NOT EXISTS` (first run only).
4. `MERGE` on `row_key`: update changed rows, insert new rows, and **delete rows no longer in scope** (e.g. first calls before May 2026). The table always matches the current definition.
5. Unit tests (`assert_true`).
6. Final selection: a B2C sample of 100 per country and customer status, read from the table (no write).

## Customer status (at the first call)
| CustomerStatus | Rule (evaluated in order) |
|---|---|
| Existing | Has a contract active on the call date (already inflowed) |
| New | No active contract, but an inflow or Power contract starts 0 to `new_customer_days` (30) days after the call |
| Churning | Had contracts that had all ended by the call date |
| Other | customer with none of the above (e.g. first contract more than 30 days later, or no contract found) |

A churned customer who comes back within 30 days counts as **New** (winback).

## Source tables
| Table | Schema | Used for |
|---|---|---|
| v_factinquirycall | glall_gold | Calls, RecallInXDays, answered flags, BrandCustomerKey |
| v_dimqueue | glall_gold | B2C/B2B queue filter, CallCustomerType |
| v_dimcall | glall_gold | Hashed phone (CustomerPhoneNumber) |
| v_puzzel_call_events | glall_silver | Call start time, rec_id, Session ID (`internal_iq_session_id`) |
| v_puzzel_call_events_pii | glall_silver | Real phone number. **PII** |
| v_factcontractbase | glall_gold | Contracts, Inflow, start/end dates |
| v_dimproduct | glall_gold | Title, ProductType (VAS), ProductGroup |
| v_dimsaleschannel | glall_gold | SalesChannel, SalesPartner, SalesProject of the Power contract |
| v_dimcontract | glall_gold | Billing join keys |
| v_dimbilling | glall_gold | Payment method |
| v_dimcustomer | glall_gold | AgeGroup, CustomerKeySource |
| v_factonlineloginevent | glall_gold | Digital logins (ForgeRock LOGIN_SUCCESS / TOKEN_REFRESHED) |

## Output columns
| Column | Description |
|---|---|
| row_key | UQ. MD5 of BrandKey + PhoneNumberHash |
| BrandKey | fims / sems / nofs |
| CallCustomerType | Consumer (B2C) / Enterprise (B2B), from the first call's queue |
| BrandCustomerKey | Matched customer |
| PhoneNumber | Real phone, E.164 digits without + (e.g. 46728633882). **PII** |
| PhoneNumberValid | Matches the SE / NO / FI numbering plan |
| SessionID | Puzzel `internal_iq_session_id` of the first call |
| CustomerStatus | New / Existing / Churning / Other / Unknown |
| FirstCallTime | Start of the first call that was followed by a recall |
| FirstRecallTime | When the customer actually called back after the first call |
| FirstRecallInXDays | Whole calendar days from the first call to the first recall (0 = same day) |
| LastRecallTime | Latest actual callback |
| AgeGroup | From v_dimcustomer |
| ProductGroup, PowerProduct | Main Power contract (v_dimproduct ProductGroup / Title) |
| VASProduct | VAS titles active on the reference date (see Step 4) |
| SalesChannel, SalesPartner, SalesProject | Sales channel, sales point (team) and sales project of the main Power contract, from v_dimsaleschannel |
| PaymentMethod | Current InvoiceChannelType |
| ChannelSwitcher | 1 = had a digital login before the first call, else 0 |
| LastDigitalActiveTimestamp | Last digital login before the first call |
| PhoneNumberHash, FirstCallDateKey | Technical: key and clustering |
| data_hash, created_at, updated_at | Change detection and load timestamps |

## Change log
| Date | Author | Change |
|---|---|---|
| 2026-11-19 | Lisa | Initial version (business definition of 2026-11-19) |
| 2026-11-23 | Lisa | Added SalesChannel, SalesPartner (sales point) and SalesProject from v_dimsaleschannel, taken from the main Power contract |

## Parameters
The widgets are bound to session variables (`v_*`) in the next cell, because temp views cannot use `:param` markers. Re-run that cell after changing a widget.
* `start_datekey`: first call date (INT yyyyMMdd), default **20260501**
* `recall_window_days`: first recall within 0 to this many days, default **30**
* `new_customer_days`: 'New' if an inflow / Power contract starts within this many days **after** the call, default **30**

In [0]:
%sql
CREATE WIDGET TEXT start_datekey      DEFAULT '20260501';
CREATE WIDGET TEXT recall_window_days DEFAULT '30';
CREATE WIDGET TEXT new_customer_days  DEFAULT '30';

In [0]:
%sql
-- Session variables are read late by the temp views below, so re-run this cell after changing a widget.
DECLARE OR REPLACE VARIABLE v_start_datekey      BIGINT;
DECLARE OR REPLACE VARIABLE v_recall_window_days INT;
DECLARE OR REPLACE VARIABLE v_new_customer_days  INT;

SET VAR v_start_datekey      = CAST(:start_datekey      AS BIGINT);
SET VAR v_recall_window_days = CAST(:recall_window_days AS INT);
SET VAR v_new_customer_days  = CAST(:new_customer_days  AS INT);

SELECT v_start_datekey, v_recall_window_days, v_new_customer_days;

## Step 1: Recall calls (`rl_calls`)
* **Trigger calls:** B2C/B2B queue calls from fims/sems/nofs since `start_datekey` with `RecallInXDays` between 0 and `recall_window_days` (30). `RecallInXDays` comes from FactInquiryCall: days from an answered, non-callout call to the next answered call from the same phone.
* **`RecallTime`:** the actual callback is the first later answered call (any queue) from the same hashed phone, on the day `RecallInXDays` points to. A rare callout can slip in because `has_callout` is not in the fact.
* **`SessionID`:** `internal_iq_session_id` of the call's first inbound Puzzel event.
* **`CallCustomerType`:** `HK.B2C` -> Enterprise, other `%B2C%` -> Consumer (B2C), else Enterprise.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW rl_calls AS
WITH ev AS (
  -- first inbound event per call (dedup)
  SELECT brandkey, call_id, rec_id, dte_start, internal_iq_session_id
  FROM cosdpprd.glall_silver.v_puzzel_call_events
  WHERE event_type = 'i'
    AND _is_deleted = false
    AND _is_active  = true
  QUALIFY ROW_NUMBER() OVER (PARTITION BY brandkey, call_id ORDER BY dte_start, rec_id) = 1
),
answered AS (
  -- answered calls in any queue = possible callbacks (same rule FactInquiryCall uses for RecallInXDays)
  SELECT
    f.BrandKey,
    dcl.CustomerPhoneNumber AS PhoneNumberHash,
    f.BrandCallKey,
    MIN(COALESCE(e.dte_start, to_timestamp(CAST(f.StartDateKey AS STRING), 'yyyyMMdd'))) AS dte_start
  FROM cosdpprd.glall_gold.v_factinquirycall f
  JOIN cosdpprd.glall_gold.v_dimcall dcl ON dcl.BrandCallKey = f.BrandCallKey
  LEFT JOIN ev e ON e.call_id = f.CallKeySource AND e.brandkey = f.BrandKey
  WHERE f.BrandKey IN ('fims', 'sems', 'nofs')
    AND f.StartDateKey >= v_start_datekey
    AND f.AgentKey IS NOT NULL
    AND (f.AnsweredExCallbackAgent = 1 OR f.AnsweredCallbackAgent = 1)
    AND dcl.CustomerPhoneNumber IS NOT NULL
  GROUP BY ALL
),
trig AS (
  -- calls followed by a recall within 0-30 days (B2C/B2B queues)
  SELECT
    f.BrandKey,
    dcl.CustomerPhoneNumber AS PhoneNumberHash,
    f.BrandCallKey,
    f.BrandCustomerKey,
    f.RecallInXDays,
    q.QueueTitle,
    CASE
      WHEN q.QueueTitle LIKE '%HK.B2C%' THEN 'Enterprise'
      WHEN q.QueueTitle LIKE '%B2C%'    THEN 'Consumer'
      ELSE 'Enterprise'
    END AS CallCustomerType,
    i.rec_id,
    i.internal_iq_session_id AS SessionID,
    COALESCE(i.dte_start, to_timestamp(CAST(f.StartDateKey AS STRING), 'yyyyMMdd')) AS dte_start
  FROM cosdpprd.glall_gold.v_factinquirycall f
  JOIN cosdpprd.glall_gold.v_dimqueue q   ON q.BrandQueueKey  = f.BrandQueueKey
  JOIN cosdpprd.glall_gold.v_dimcall  dcl ON dcl.BrandCallKey = f.BrandCallKey
  LEFT JOIN ev i ON i.call_id = f.CallKeySource AND i.brandkey = f.BrandKey
  WHERE f.BrandKey IN ('fims', 'sems', 'nofs')
    AND f.StartDateKey >= v_start_datekey
    AND (q.QueueTitle LIKE '%B2C.%' OR q.QueueTitle LIKE '%B2B.%')
    AND f.RecallInXDays BETWEEN 0 AND v_recall_window_days
    AND dcl.CustomerPhoneNumber IS NOT NULL
)
SELECT
  t.BrandKey,
  t.PhoneNumberHash,
  t.BrandCallKey,
  t.BrandCustomerKey,
  t.RecallInXDays,
  t.QueueTitle,
  t.CallCustomerType,
  t.rec_id,
  t.SessionID,
  t.dte_start,
  MIN(a.dte_start) AS RecallTime   -- actual callback time
FROM trig t
LEFT JOIN answered a
  ON  a.BrandKey        = t.BrandKey
  AND a.PhoneNumberHash = t.PhoneNumberHash
  AND a.BrandCallKey   <> t.BrandCallKey
  AND a.dte_start       > t.dte_start
  AND datediff(to_date(a.dte_start), to_date(t.dte_start)) = t.RecallInXDays
GROUP BY ALL;

## Step 2: One row per recaller (`rl_recallers`)
Grain: BrandKey + PhoneNumberHash. First-call attributes use `MIN_BY(..., dte_start)` with `BrandCallKey` as tie-breaker. `FirstRecallTime` = callback after the first call, `FirstRecallInXDays` = whole calendar days from the first call to that callback (FactInquiryCall `RecallInXDays` of the first call, 0 = same day), `LastRecallTime` = latest callback. `BrandCustomerKey` = earliest non-null customer key.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW rl_recallers AS
SELECT
  BrandKey,
  PhoneNumberHash,
  MIN(dte_start)                                            AS FirstCallTime,
  MIN_BY(BrandCallKey,     struct(dte_start, BrandCallKey)) AS FirstBrandCallKey,
  MIN_BY(rec_id,           struct(dte_start, BrandCallKey)) AS first_rec_id,
  MIN_BY(SessionID,        struct(dte_start, BrandCallKey)) AS SessionID,
  MIN_BY(CallCustomerType, struct(dte_start, BrandCallKey)) AS CallCustomerType,
  MIN_BY(RecallTime,       struct(dte_start, BrandCallKey)) AS FirstRecallTime,
  MIN_BY(RecallInXDays,    struct(dte_start, BrandCallKey)) AS FirstRecallInXDays,   -- days from first call to first recall
  MAX(RecallTime)                                           AS LastRecallTime,
  MIN_BY(BrandCustomerKey, struct(BrandCustomerKey IS NULL, dte_start)) AS BrandCustomerKey  -- non-null first
FROM rl_calls
GROUP BY ALL;

## Step 3: Contracts of the recaller's customer (`rl_contracts`)
All contracts of the matched `BrandCustomerKey`, with `Inflow`, product Title / ProductType / ProductGroup, `SalesChannel` / `SalesPartner` / `SalesProject` from v_dimsaleschannel and billing keys. `call_date` = `to_date(FirstCallTime)`.

`ContractEndDate` is **exclusive** and far-future when open. A contract is active on date d when `ContractStartDate <= d AND ContractEndDate > d`.

## Step 4: Customer status (`rl_status`)
The status rules are in the header table. Any contract category counts for Existing and Churning. For New, the contract must be an inflow (`Inflow = 1`) or a Power contract starting 0 to `new_customer_days` after the call date.

`ref_date` is the date the products and the Power contract are read on:
* Existing: call date
* New: start of the new contract
* Churning: last day before the relationship ended
* Unknown / Other: none

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW rl_status AS
WITH agg AS (
  SELECT
    r.BrandKey,
    r.PhoneNumberHash,
    r.BrandCustomerKey,
    to_date(r.FirstCallTime) AS call_date,
    COALESCE(BOOL_OR(c.ContractStartDate <= c.call_date AND c.ContractEndDate > c.call_date), false) AS ActiveAtCall,
    MIN(CASE WHEN (c.Inflow = 1 OR c.ProductCategory = 'Power')
              AND c.ContractStartDate >= c.call_date
              AND c.ContractStartDate <= date_add(c.call_date, v_new_customer_days)
             THEN c.ContractStartDate END) AS FirstNewStartDate,
    MAX(CASE WHEN c.ContractStartDate <= c.call_date THEN c.ContractEndDate END) AS LastEndBeforeCall
  FROM rl_recallers r
  LEFT JOIN rl_contracts c ON c.BrandKey = r.BrandKey AND c.PhoneNumberHash = r.PhoneNumberHash
  GROUP BY ALL
)
SELECT
  BrandKey,
  PhoneNumberHash,
  CASE
    WHEN ActiveAtCall                   THEN 'Existing'
    WHEN FirstNewStartDate IS NOT NULL  THEN 'New'
    WHEN LastEndBeforeCall IS NOT NULL  THEN 'Churning'
    ELSE 'Other'
  END AS CustomerStatus,
  CASE CustomerStatus
    WHEN 'Existing' THEN call_date
    WHEN 'New'      THEN FirstNewStartDate
    WHEN 'Churning' THEN date_sub(LastEndBeforeCall, 1)
  END AS ref_date
FROM agg;

## Step 5: Products (`rl_primary_contract`, `rl_vas`)
* **Primary Power contract:** a Power contract active on `ref_date`. If none, the latest one started before the call, otherwise the nearest one after. It gives `PowerProduct` (Title), `ProductGroup`, `SalesChannel`, `SalesPartner`, `SalesProject` (the sales channel the Power contract was sold through) and the billing keys.
* **`VASProduct`:** comma-separated titles of VAS products (`ProductType` VAS / VAS Other) active on `ref_date`.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW rl_primary_contract AS
SELECT
  c.BrandKey,
  c.PhoneNumberHash,
  c.BrandContractKey AS PrimaryBrandContractKey,
  c.Title            AS PowerProduct,
  c.ProductGroup,
  c.SalesChannel,
  c.SalesPartner,
  c.SalesProject,
  c.ContractKeySource,
  c.CustomerKeySource
FROM rl_contracts c
JOIN rl_status s ON s.BrandKey = c.BrandKey AND s.PhoneNumberHash = c.PhoneNumberHash
WHERE c.ProductCategory = 'Power'
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY c.BrandKey, c.PhoneNumberHash
  ORDER BY
    CASE WHEN c.ContractStartDate <= s.ref_date AND c.ContractEndDate > s.ref_date THEN 0
         WHEN c.ContractStartDate <= c.call_date THEN 1
         ELSE 2 END,
    abs(datediff(c.ContractStartDate, COALESCE(s.ref_date, c.call_date))),
    c.BrandContractKey
) = 1;

CREATE OR REPLACE TEMP VIEW rl_vas AS
SELECT
  s.BrandKey,
  s.PhoneNumberHash,
  NULLIF(array_join(array_sort(collect_set(
    CASE WHEN c.ProductType IN ('VAS', 'VAS Other')
          AND c.ContractStartDate <= s.ref_date AND c.ContractEndDate > s.ref_date THEN c.Title END
  )), ', '), '') AS VASProduct
FROM rl_status s
LEFT JOIN rl_contracts c ON c.BrandKey = s.BrandKey AND c.PhoneNumberHash = s.PhoneNumberHash
GROUP BY ALL;

## Step 6: Payment method (`rl_payment`)
`PaymentMethod` = `v_dimbilling.InvoiceChannelType`, taken from the primary contract (composite key `BrandKey + ContractKeySource + CustomerKeySource`). If there is none, any active billing row of the customer is used. **Current** value, because v_dimbilling has no history.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW rl_payment AS
WITH contract_pm AS (
  SELECT p.BrandKey, p.PhoneNumberHash, b.InvoiceChannelType
  FROM rl_primary_contract p
  JOIN cosdpprd.glall_gold.v_dimbilling b
    ON  b.BrandKey          = p.BrandKey
    AND b.ContractKeySource = p.ContractKeySource
    AND b.CustomerKeySource = p.CustomerKeySource
  QUALIFY ROW_NUMBER() OVER (PARTITION BY p.BrandKey, p.PhoneNumberHash
                             ORDER BY b.isActive DESC, b.BillingKey DESC) = 1
),
customer_pm AS (
  SELECT r.BrandKey, r.PhoneNumberHash, b.InvoiceChannelType
  FROM rl_recallers r
  JOIN cosdpprd.glall_gold.v_dimcustomer dc ON dc.BrandCustomerKey = r.BrandCustomerKey
  JOIN cosdpprd.glall_gold.v_dimbilling  b
    ON  b.BrandKey          = dc.BrandKey
    AND b.CustomerKeySource = TRY_CAST(dc.CustomerKeySource AS BIGINT)
  WHERE b.isActive = 1
  QUALIFY ROW_NUMBER() OVER (PARTITION BY r.BrandKey, r.PhoneNumberHash
                             ORDER BY b.BillingKey DESC) = 1
)
SELECT
  r.BrandKey,
  r.PhoneNumberHash,
  COALESCE(c.InvoiceChannelType, cu.InvoiceChannelType) AS PaymentMethod
FROM rl_recallers r
LEFT JOIN contract_pm c  ON c.BrandKey  = r.BrandKey AND c.PhoneNumberHash  = r.PhoneNumberHash
LEFT JOIN customer_pm cu ON cu.BrandKey = r.BrandKey AND cu.PhoneNumberHash = r.PhoneNumberHash;

## Step 7: Digital activity (`rl_digital`)
`LastDigitalActiveTimestamp` = the last event in `v_factonlineloginevent` **before** the first call. The event is matched on BrandKey + CustomerKeySource from v_dimcustomer. `ChannelSwitcher` = 1 if such an event exists, else 0.

**Notes:**
* The fact includes ForgeRock `LOGIN_SUCCESS` **and** `TOKEN_REFRESHED` events.
* Login data starts in Dec 2024, and no lookback window is applied, so any earlier login counts.
* Unknown customers (no match) always get 0.
* Login timestamps are UTC. Check that Puzzel `dte_start` is in the same time zone.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW rl_digital AS
SELECT
  r.BrandKey,
  r.PhoneNumberHash,
  MAX(l.EventTimestamp) AS LastDigitalActiveTimestamp
FROM rl_recallers r
JOIN cosdpprd.glall_gold.v_dimcustomer dc ON dc.BrandCustomerKey = r.BrandCustomerKey
JOIN cosdpprd.glall_gold.v_factonlineloginevent l
  ON  l.BrandKey          = dc.BrandKey
  AND l.CustomerKeySource = TRY_CAST(dc.CustomerKeySource AS BIGINT)
  AND l.EventTimestamp    < r.FirstCallTime
GROUP BY ALL;

## Step 8: Final staging (`rl_final`)
Joins everything per recaller and adds the real phone number. **PII:** `v_puzzel_call_events_pii.source` for the first call's `rec_id`, joined on brand + rec_id. It is normalized to E.164 digits without `+` (e.g. 46728633882) and validated per country:

| Country | Rule |
|---|---|
| Sweden | `46` + 7 to 9 digits |
| Norway | `47` + 8 digits |
| Finland | `358` + 5 to 12 digits |
| Other | 8 to 15 digits |

The view also adds `row_key` = MD5(BrandKey|PhoneNumberHash) and `data_hash` = SHA-256 of all non-key columns.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW rl_final AS
WITH base AS (
  SELECT
    MD5(CONCAT_WS('|', r.BrandKey, r.PhoneNumberHash))   AS row_key,
    r.BrandKey,
    r.CallCustomerType,
    r.BrandCustomerKey,
    pi.PhoneNumber,                                        -- PII
    COALESCE(pi.PhoneNumberValid, false)                  AS PhoneNumberValid,
    r.SessionID,
    s.CustomerStatus,
    r.FirstCallTime,
    r.FirstRecallTime,
    r.FirstRecallInXDays,
    r.LastRecallTime,
    dc.AgeGroup,
    pc.ProductGroup,
    pc.PowerProduct,
    v.VASProduct,
    pc.SalesChannel,
    pc.SalesPartner,
    pc.SalesProject,
    pm.PaymentMethod,
    CASE WHEN d.LastDigitalActiveTimestamp IS NOT NULL THEN 1 ELSE 0 END AS ChannelSwitcher,
    d.LastDigitalActiveTimestamp,
    r.PhoneNumberHash,
    CAST(date_format(r.FirstCallTime, 'yyyyMMdd') AS INT) AS FirstCallDateKey
  FROM rl_recallers r
  LEFT JOIN (
    -- Phone to E.164 digits without '+', validated per country numbering plan
    SELECT
      brandkey,
      rec_id,
      PhoneNumber,
      CASE
        WHEN PhoneNumber IS NULL            THEN false
        WHEN startswith(PhoneNumber, '46')  THEN PhoneNumber RLIKE '^46[1-9][0-9]{6,8}$'     -- SE
        WHEN startswith(PhoneNumber, '47')  THEN PhoneNumber RLIKE '^47[2-9][0-9]{7}$'       -- NO
        WHEN startswith(PhoneNumber, '358') THEN PhoneNumber RLIKE '^358[1-9][0-9]{4,11}$'   -- FI
        ELSE PhoneNumber RLIKE '^[1-9][0-9]{7,14}$'                                          -- foreign
      END AS PhoneNumberValid
    FROM (
      SELECT
        brandkey,
        rec_id,
        -- drop a trunk 0 written after the country code: +46 (0)72... / 3580401...
        CASE WHEN startswith(n, concat(cc, '0')) THEN concat(cc, substr(n, length(cc) + 2)) ELSE n END AS PhoneNumber
      FROM (
        SELECT
          brandkey,
          rec_id,
          cc,
          CASE
            WHEN d IS NULL OR d = ''                               THEN NULL                       -- hidden caller
            WHEN startswith(raw, '+')                              THEN d                          -- international as given
            WHEN startswith(d, '00')                               THEN substr(d, 3)               -- 0046... -> 46...
            WHEN startswith(d, cc) AND length(d) >= length(cc) + 7 THEN d                          -- already 46... without +
            WHEN startswith(d, '0')                                THEN concat(cc, substr(d, 2))   -- SE/FI trunk 0
            WHEN length(d) BETWEEN 7 AND 9                         THEN concat(cc, d)              -- NO 8 digits / no trunk 0
            ELSE d
          END AS n
        FROM (
          SELECT
            brandkey,
            rec_id,
            trim(source)                         AS raw,
            regexp_replace(source, '[^0-9]', '') AS d,
            CASE brandkey WHEN 'sems' THEN '46' WHEN 'nofs' THEN '47' WHEN 'fims' THEN '358' END AS cc
          FROM cosdpprd.glall_silver.v_puzzel_call_events_pii
          WHERE _is_deleted = false
            AND _is_active  = true
          QUALIFY ROW_NUMBER() OVER (PARTITION BY brandkey, rec_id ORDER BY _load_date DESC) = 1
        )
      )
    )
  ) pi ON pi.rec_id = r.first_rec_id AND pi.brandkey = r.BrandKey   -- brand-scoped join is required
  LEFT JOIN cosdpprd.glall_gold.v_dimcustomer dc ON dc.BrandCustomerKey = r.BrandCustomerKey
  LEFT JOIN rl_status           s  ON s.BrandKey  = r.BrandKey AND s.PhoneNumberHash  = r.PhoneNumberHash
  LEFT JOIN rl_primary_contract pc ON pc.BrandKey = r.BrandKey AND pc.PhoneNumberHash = r.PhoneNumberHash
  LEFT JOIN rl_vas              v  ON v.BrandKey  = r.BrandKey AND v.PhoneNumberHash  = r.PhoneNumberHash
  LEFT JOIN rl_payment          pm ON pm.BrandKey = r.BrandKey AND pm.PhoneNumberHash = r.PhoneNumberHash
  LEFT JOIN rl_digital          d  ON d.BrandKey  = r.BrandKey AND d.PhoneNumberHash  = r.PhoneNumberHash
)
SELECT
  *,
  sha2(to_json(struct(* EXCEPT (row_key, BrandKey, PhoneNumberHash))), 256) AS data_hash
FROM base;

## Create the output table (first run only)
`CREATE TABLE IF NOT EXISTS` (never CREATE OR REPLACE). Liquid clustered on `(BrandKey, FirstCallDateKey)`. `PhoneNumber` is tagged `pii_level = high`.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cosdpprd.glall_user.recallerlist (
  row_key                    STRING    NOT NULL COMMENT 'UQ. MD5 of BrandKey|PhoneNumberHash, MERGE key',
  BrandKey                   STRING    COMMENT 'Brand: fims / sems / nofs',
  CallCustomerType           STRING    COMMENT 'Consumer (B2C) / Enterprise (B2B) from the first call queue',
  BrandCustomerKey           STRING    COMMENT 'Matched customer key',
  PhoneNumber                STRING    COMMENT 'Real caller phone number, E.164 digits without + (e.g. 46728633882). PII',
  PhoneNumberValid           BOOLEAN   COMMENT 'PhoneNumber matches the SE (46) / NO (47) / FI (358) numbering plan, or E.164 length for foreign numbers',
  SessionID                  STRING    COMMENT 'Puzzel internal_iq_session_id of the first call',
  CustomerStatus             STRING    COMMENT 'New / Existing / Churning / Other / Unknown at the first call',
  FirstCallTime              TIMESTAMP COMMENT 'Start of the first call that was followed by a recall within 0-30 days',
  FirstRecallTime            TIMESTAMP COMMENT 'Time the customer actually called back after the first call',
  FirstRecallInXDays         INT       COMMENT 'Whole calendar days from the first call to the first recall (0 = same day)',
  LastRecallTime             TIMESTAMP COMMENT 'Time of the latest actual callback',
  AgeGroup                   STRING    COMMENT 'v_dimcustomer.AgeGroup',
  ProductGroup               STRING    COMMENT 'v_dimproduct.ProductGroup of the primary Power contract',
  PowerProduct               STRING    COMMENT 'Product title of the primary Power contract',
  VASProduct                 STRING    COMMENT 'Comma-separated VAS product titles active on the reference date',
  SalesChannel               STRING    COMMENT 'v_dimsaleschannel.SalesChannel of the primary Power contract',
  SalesPartner               STRING    COMMENT 'v_dimsaleschannel.SalesPartner (sales point / team) of the primary Power contract',
  SalesProject               STRING    COMMENT 'v_dimsaleschannel.SalesProject (project / activity behind the sale) of the primary Power contract',
  PaymentMethod              STRING    COMMENT 'Current InvoiceChannelType (not as of call time)',
  ChannelSwitcher            INT       COMMENT '1 = had a digital login (v_factonlineloginevent) before the first call, else 0',
  LastDigitalActiveTimestamp TIMESTAMP COMMENT 'Last digital login before the first call',
  PhoneNumberHash            STRING    COMMENT 'Hashed caller phone number (v_dimcall.CustomerPhoneNumber)',
  FirstCallDateKey           INT       COMMENT 'FirstCallTime as INT yyyyMMdd',
  data_hash                  STRING    COMMENT 'SHA-256 of all non-key attributes, used to detect changes',
  created_at                 TIMESTAMP COMMENT 'When the row was first inserted',
  updated_at                 TIMESTAMP COMMENT 'When the row was last updated'
)
CLUSTER BY (BrandKey, FirstCallDateKey)
COMMENT 'Recaller list (Nordic Customer, owner Lisa). One row per recaller (BrandKey + hashed phone) whose first recall is within 0-30 days, first calls from 2026-05-01. Built by notebook Recaller List via MERGE on row_key. Contains PII (PhoneNumber).';

ALTER TABLE cosdpprd.glall_user.recallerlist ALTER COLUMN PhoneNumber SET TAGS ('pii_level' = 'high');

## Merge staging into the output table
* Matched and `data_hash` changed: update
* Not matched: insert
* **Not matched by source: delete.** Rows that are no longer in scope (e.g. first calls before `start_datekey`, or no longer a 0-30 day recaller) are removed, so the table always matches the current definition.

In [0]:
%sql
MERGE INTO cosdpprd.glall_user.recallerlist AS target
USING rl_final AS source
ON target.row_key = source.row_key
WHEN MATCHED AND target.data_hash <> source.data_hash THEN UPDATE SET
  target.BrandKey                   = source.BrandKey,
  target.CallCustomerType           = source.CallCustomerType,
  target.BrandCustomerKey           = source.BrandCustomerKey,
  target.PhoneNumber                = source.PhoneNumber,
  target.PhoneNumberValid           = source.PhoneNumberValid,
  target.SessionID                  = source.SessionID,
  target.CustomerStatus             = source.CustomerStatus,
  target.FirstCallTime              = source.FirstCallTime,
  target.FirstRecallTime            = source.FirstRecallTime,
  target.FirstRecallInXDays         = source.FirstRecallInXDays,
  target.LastRecallTime             = source.LastRecallTime,
  target.AgeGroup                   = source.AgeGroup,
  target.ProductGroup               = source.ProductGroup,
  target.PowerProduct               = source.PowerProduct,
  target.VASProduct                 = source.VASProduct,
  target.SalesChannel               = source.SalesChannel,
  target.SalesPartner               = source.SalesPartner,
  target.SalesProject               = source.SalesProject,
  target.PaymentMethod              = source.PaymentMethod,
  target.ChannelSwitcher            = source.ChannelSwitcher,
  target.LastDigitalActiveTimestamp = source.LastDigitalActiveTimestamp,
  target.PhoneNumberHash            = source.PhoneNumberHash,
  target.FirstCallDateKey           = source.FirstCallDateKey,
  target.data_hash                  = source.data_hash,
  target.updated_at                 = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT (
  row_key, BrandKey, CallCustomerType, BrandCustomerKey, PhoneNumber, PhoneNumberValid,
  SessionID, CustomerStatus, FirstCallTime, FirstRecallTime, FirstRecallInXDays, LastRecallTime, AgeGroup,
  ProductGroup, PowerProduct, VASProduct, SalesChannel, SalesPartner, SalesProject, PaymentMethod, ChannelSwitcher,
  LastDigitalActiveTimestamp, PhoneNumberHash, FirstCallDateKey, data_hash, created_at, updated_at
) VALUES (
  source.row_key, source.BrandKey, source.CallCustomerType, source.BrandCustomerKey, source.PhoneNumber, source.PhoneNumberValid,
  source.SessionID, source.CustomerStatus, source.FirstCallTime, source.FirstRecallTime, source.FirstRecallInXDays, source.LastRecallTime, source.AgeGroup,
  source.ProductGroup, source.PowerProduct, source.VASProduct, source.SalesChannel, source.SalesPartner, source.SalesProject, source.PaymentMethod, source.ChannelSwitcher,
  source.LastDigitalActiveTimestamp, source.PhoneNumberHash, source.FirstCallDateKey, source.data_hash, CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()
)
WHEN NOT MATCHED BY SOURCE THEN DELETE;

## Unit tests
The tests check that `row_key` is unique, PhoneNumberHash is not null, callbacks come after the first call, no first call is before `start_datekey`, `FirstRecallInXDays` is between 0 and `recall_window_days`, and CustomerStatus has only the expected values. The assert cell fails the run if any test fails.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW rl_unit_test AS
SELECT
  'RowKeyDuplicates'          AS KPI,
  COALESCE(COUNT(*), 0)       AS ActualValue,
  0                           AS ExpectedValue,
  ActualValue = ExpectedValue AS test_result
FROM (
  SELECT row_key, COUNT(*) AS n_records
  FROM cosdpprd.glall_user.recallerlist
  GROUP BY ALL
  HAVING n_records > 1
)
UNION ALL
SELECT 'PhoneNumberHashNull', COUNT(*), 0, COUNT(*) = 0
FROM cosdpprd.glall_user.recallerlist
WHERE PhoneNumberHash IS NULL
UNION ALL
SELECT 'RecallTimeNotAfterFirstCall', COUNT(*), 0, COUNT(*) = 0
FROM cosdpprd.glall_user.recallerlist
WHERE FirstRecallTime <= FirstCallTime
UNION ALL
SELECT 'FirstCallBeforeStartDate', COUNT(*), 0, COUNT(*) = 0
FROM cosdpprd.glall_user.recallerlist
WHERE FirstCallDateKey < v_start_datekey
UNION ALL
SELECT 'FirstRecallInXDaysOutOfWindow', COUNT(*), 0, COUNT(*) = 0
FROM cosdpprd.glall_user.recallerlist
WHERE FirstRecallInXDays IS NULL OR FirstRecallInXDays NOT BETWEEN 0 AND v_recall_window_days
UNION ALL
SELECT 'UnexpectedCustomerStatus', COUNT(*), 0, COUNT(*) = 0
FROM cosdpprd.glall_user.recallerlist
WHERE CustomerStatus IS NULL OR CustomerStatus NOT IN ('New', 'Existing', 'Churning', 'Other');

SELECT * FROM rl_unit_test;

In [0]:
%sql
SELECT
  KPI,
  COALESCE(assert_true(test_result, 'Some unit tests failed. Please, validate the logic implemented in the notebook.'), 'PASS') AS assertion
FROM rl_unit_test;

## Final selection (read only, no write)
This reads from `cosdpprd.glall_user.recallerlist`:
* **B2C** only (`CallCustomerType = 'Consumer'`)
* Status groups **New / Existing / Churning**
* **100 per country and status**: 3 countries x 3 statuses = up to 900 rows

The sample is pseudo-random but repeatable (ordered by `xxhash64(row_key)`). A group with fewer than 100 recallers returns all of them. The first cell shows the group sizes. The second returns the sample, which **contains PII**.

In [0]:
%sql
SELECT
  CASE BrandKey WHEN 'fims' THEN 'Finland' WHEN 'sems' THEN 'Sweden' WHEN 'nofs' THEN 'Norway' END AS Country,
  CustomerStatus,
  COUNT(*)            AS recallers_in_table,
  LEAST(COUNT(*), 100) AS rows_in_sample
FROM cosdpprd.glall_user.recallerlist
WHERE CallCustomerType = 'Consumer'                          -- B2C
  AND CustomerStatus IN ('New', 'Existing', 'Churning')
GROUP BY ALL
ORDER BY Country, CustomerStatus;

In [0]:
%sql
SELECT
  CASE BrandKey WHEN 'fims' THEN 'Finland' WHEN 'sems' THEN 'Sweden' WHEN 'nofs' THEN 'Norway' END AS Country,
  BrandCustomerKey,
  PhoneNumber,                        -- PII
  SessionID,
  CustomerStatus,
  FirstCallTime,
  FirstRecallTime,
  FirstRecallInXDays,
  LastRecallTime,
  AgeGroup,
  ProductGroup,
  PowerProduct,
  VASProduct,
  SalesChannel,
  SalesPartner,
  SalesProject,
  PaymentMethod,
  ChannelSwitcher,
  LastDigitalActiveTimestamp
FROM cosdpprd.glall_user.recallerlist
WHERE CallCustomerType = 'Consumer'                          -- B2C
  AND CustomerStatus IN ('New', 'Existing', 'Churning')
QUALIFY ROW_NUMBER() OVER (PARTITION BY BrandKey, CustomerStatus ORDER BY xxhash64(row_key)) <= 100
ORDER BY Country, CustomerStatus, FirstCallTime;